# PhysAR-IDS -- NB4d (v6): Attack 2 with a physical-layer attacker bound by the power balance

**Why (reviewer issue 3).** Every attack so far could change only the 60 cyber features. The reviewer's objection: a 0% evasion of ransomware / reverseshell then follows from the threat model, not from physics, and an FDI attacker by definition *can* falsify measurements. SHAP (NB-SHAP) showed physical features carry 11-33% of the attribution, and that removing them costs ransomware / reverseshell about 0.15 F1 (others under 0.05). This notebook lets the attacker change the physical measurements too, **within physically consistent limits**, and asks whether the physical layer then protects those classes.

**The physical limit used here is the power balance, checked on the real data.** In every class of the dataset (benign and all five attacks, FDI included) the apparent power satisfies  
`S = sqrt(P^2 + Q^2) = V_A*I_A + V_B*I_B + V_C*I_C`  
to within about +/-0.06% (1st-99th percentile of S / sum(V*I): 0.9993 - 1.0006). A forged measurement set that breaks this identity is physically inconsistent. The balance-constrained attacker therefore rescales (P, Q) after every step so that S / sum(V*I) stays at the row's own original value; the *direction* (power factor) of (P, Q) remains the agent's choice.

**Threat model per setting** (black-box hard-label oracle, tuned XGBoost, same PPO as NB4: RecurrentPPO, 50,000 steps, 10 queries per episode, evasion reward +10):
* **Physical channel:** the 9 measurements `V_A, V_B, V_C, I_A, I_B, I_C, ActivePower, ReactivePower, Theta`. Budget = cumulative **relative** change of each measurement from the row's own original value (`eps_rel`, e.g. 5%), because the nine PLCs differ by orders of magnitude in current and power, so a budget in global standard deviations would be meaningless for the small ones. Clipped to the training range.
* **Derived features** `voltage_imbalance, current_imbalance, power_factor` are **recomputed** from the perturbed measurements (never perturbed independently). `Freq`, `BreakerStatus`, `freq_deviation` are constant in this dataset.
* **Temporal features** (`d_*`, `roll_*`, 18 columns) are **held at their observed values**: they depend on the previous samples, which a single-row attacker does not control. This is a stated limitation and favours the attacker slightly (the frozen values cannot reveal the change).
* **Cyber channel** (optional): the 60 cyber features with the NB4 budget in standard deviations.

**Settings** (edit `SETTINGS`): A physical 5%, no balance; B physical 5%, balance; C cyber 0.5 sigma + physical 5%, balance; D physical 1%, balance. Setting C is the informative one: at 0.5 sigma the cyber-only attacker gets 0.000 on ransomware / reverseshell (NB4b), so any evasion there is due to the physical channel. A vs B shows what the balance constraint costs the attacker. A uniform-random attacker with the same limits is the baseline. Each PPO agent takes ~7-9 minutes (4 settings x 5 seeds ~ 2.5-3 h, resumable per agent).

**Reading the result.** If ransomware / reverseshell stay near 0 under B and D, the physical layer does protect them against this attacker. If C flips them, the cyber threshold of ~0.5-1 sigma drops once a small consistent physical change is allowed, and the dissertation must say that the protection depends on what the attacker can touch. Checks that make the result auditable: the real-data balance identity is re-measured on the test rows, the derived-feature recomputation is verified against the stored values, and the final adversarial rows are tested against the genuine balance band.

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings
import numpy as np, pandas as pd, joblib
import torch
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (as NB4) ----
EPS_BOUND = 0.3; MAX_STEPS = 10                    # per-step cyber clip in sigma (the policy outputs [-1, 1] and is scaled)
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EVAL_CHUNK = 8192
# cyber: cumulative budget in sigma (None = cyber channel frozen); eps_rel: cumulative relative change of each measurement; balance: enforce S = sum(V*I)
SETTINGS = {
    'A_phys5_nobalance':    dict(cyber=None, eps_rel=0.05, balance=False),
    'B_phys5_balance':      dict(cyber=None, eps_rel=0.05, balance=True),
    'C_cyber0p5_phys5_bal': dict(cyber=0.5,  eps_rel=0.05, balance=True),
    'D_phys1_balance':      dict(cyber=None, eps_rel=0.01, balance=True),
}
SETTINGS_TO_RUN = list(SETTINGS)                   # remove names to run fewer settings

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
import sb3_contrib, stable_baselines3
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME} | sb3-contrib {sb3_contrib.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data, the NB2 tuned configuration and the scaler (raw physical units are reconstructed in float64)

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train64 = np.asarray(data['X_train_scaled'], dtype=np.float64); X_test64 = np.asarray(data['X_test_scaled'], dtype=np.float64)
X_train = X_train64.astype(np.float32); X_test = X_test64.astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
scaler = joblib.load(f'standard_scaler_{PIPELINE_VERSION}.pkl'); MU = np.asarray(scaler.mean_, dtype=np.float64); SC = np.asarray(scaler.scale_, dtype=np.float64)
assert len(MU) == N_FEAT

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]; X_atk_train64 = X_train64[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te];  X_atk_test64 = X_test64[atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)

PB_NAMES = ['V_A', 'V_B', 'V_C', 'I_A', 'I_B', 'I_C', 'ActivePower', 'ReactivePower', 'Theta']      # the 9 physical measurements the attacker may change
PB = np.array([FEATURE_COLS.index(c) for c in PB_NAMES]); DER = np.array([FEATURE_COLS.index(c) for c in ['voltage_imbalance', 'current_imbalance', 'power_factor']])
N_CTRL_PHYS = len(PB)

def raw_of(Xs): return Xs[:, PB] * SC[PB] + MU[PB]                                  # float64 (n, 9) in physical units
def derive(raw):
    V, I = raw[:, 0:3], raw[:, 3:6]; Vm, Im = V.mean(1), I.mean(1)
    with np.errstate(divide='ignore', invalid='ignore'):
        vimb = np.abs(V - Vm[:, None]).max(1) / np.where(Vm == 0, np.nan, Vm); cimb = np.abs(I - Im[:, None]).max(1) / np.where(Im == 0, np.nan, Im)
        S = np.hypot(raw[:, 6], raw[:, 7]); pf = raw[:, 6] / np.where(S == 0, np.nan, S)
    return np.nan_to_num(np.stack([vimb, cimb, pf], 1))
def ratio_of(raw):                                                                   # S / sum(V*I): the power-balance residual
    VI = (raw[:, 0:3] * raw[:, 3:6]).sum(1)
    with np.errstate(divide='ignore', invalid='ignore'): return np.hypot(raw[:, 6], raw[:, 7]) / np.where(VI == 0, np.nan, VI)
def write_back(x, raw):                                                              # x (n, 93) float64, in place: measurements + recomputed derived features
    x[:, PB] = (raw - MU[PB]) / SC[PB]; x[:, DER] = (derive(raw) - MU[DER]) / SC[DER]

raw_train = raw_of(X_train64); RAW_LO, RAW_HI = raw_train.min(0), raw_train.max(0)
r_tr = ratio_of(raw_train); r_tr = r_tr[np.isfinite(r_tr)]
BAND_LO, BAND_HI = np.percentile(r_tr, [0.5, 99.5])
r_te = ratio_of(raw_of(X_test64)); r_te = r_te[np.isfinite(r_te)]
print(f'Features {N_FEAT} | cyber channel {N_CYBER} | physical channel {N_CTRL_PHYS} | evaluation rows {X_atk_test.shape} | training pool {X_atk_train.shape}')
print(f'Power balance S / sum(V*I) on the TEST rows (reconstructed from the scaled data): 1st {np.percentile(r_te, 1):.5f}, median {np.median(r_te):.5f}, 99th {np.percentile(r_te, 99):.5f}')
print(f'Genuine band used for realism checks (0.5th-99.5th percentile of TRAIN rows): [{BAND_LO:.5f}, {BAND_HI:.5f}]')
der_err = np.abs((derive(raw_of(X_test64)) - MU[DER]) / SC[DER] - X_test64[:, DER]).max()
print(f'Max |recomputed - stored| derived feature (sigma units): {der_err:.2e}')
if not SMOKE_TEST:
    assert abs(np.median(r_te) - 1.0) < 1e-3 and np.percentile(r_te, 1) > 0.998 and np.percentile(r_te, 99) < 1.002, 'power balance not reproduced -- scaler / column order problem'
    assert der_err < 1e-3, 'derived-feature recomputation does not reproduce the stored values'
print('Test sigma of each controlled measurement (global, for reference):', dict(zip(PB_NAMES, [f'{v:.3g}' for v in SC[PB]])))
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
print('Attack bursts in the evaluation set:', len(np.unique(test_burst)))
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Oracle, step function, environment and batched evaluator

In [ ]:
class Oracle:
    """Hard-label black-box oracle over the fitted XGBoost classifier."""
    def __init__(self, clf): self.clf = clf; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X)
        return self.clf.predict(X)

def act_dim(st): return (N_CYBER if st['cyber'] is not None else 0) + N_CTRL_PHYS

def make_ctx(x_orig):                                                                # per-row constants of an episode
    raw0 = raw_of(x_orig); rho0 = ratio_of(raw0); rho0 = np.where(np.isfinite(rho0), rho0, 1.0)
    return raw0, rho0

def step_fn(x_cur, x_orig, raw0, rho0, act, st):
    """Apply one attacker step in place on x_cur (n, 93) float64.  act in [-1, 1]^(act_dim).  Returns the applied move (n, act_dim):
    cyber part in sigma, physical part as a fraction of the per-row budget (so 1 = the full cumulative budget)."""
    parts = []; a = np.clip(np.asarray(act, dtype=np.float64), -1.0, 1.0)
    if st['cyber'] is not None:
        old = x_cur[:, CYBER_IDX].copy(); o = x_orig[:, CYBER_IDX]
        prop = old + a[:, :N_CYBER] * EPS_BOUND
        prop = o + np.clip(prop - o, -st['cyber'], st['cyber'])                      # cumulative budget (sigma)
        prop = np.clip(prop, np.minimum(FEAT_LO[CYBER_IDX], o), np.maximum(FEAT_HI[CYBER_IDX], o))   # observed training range (NB4c)
        x_cur[:, CYBER_IDX] = prop; parts.append((prop - old).astype(np.float32))
    eps = st['eps_rel']; ap = a[:, -N_CTRL_PHYS:]
    raw_cur = raw_of(x_cur)
    prop = raw_cur * (1.0 + ap * eps)                                                # per-step relative move <= eps
    with np.errstate(divide='ignore', invalid='ignore'): cum = np.where(raw0 != 0, prop / raw0 - 1.0, 0.0)
    prop = raw0 * (1.0 + np.clip(cum, -eps, eps))                                    # cumulative relative budget
    prop = np.clip(prop, np.minimum(RAW_LO, raw0), np.maximum(RAW_HI, raw0))         # observed training range
    if st['balance']:                                                                # keep S = rho0 * sum(V*I): rescale (P, Q), keep their direction
        VI = (prop[:, 0:3] * prop[:, 3:6]).sum(1); S = np.hypot(prop[:, 6], prop[:, 7])
        f = np.where(S > 0, rho0 * VI / np.where(S > 0, S, 1.0), 1.0); f = np.where(np.isfinite(f) & (f > 0), f, 1.0)
        prop[:, 6] *= f; prop[:, 7] *= f
    write_back(x_cur, prop)
    denom = np.maximum(np.abs(raw0) * eps, 1e-12)
    parts.append(np.clip((prop - raw_cur) / denom, -3, 3).astype(np.float32))
    return np.concatenate(parts, 1)

def penalty(applied, st):
    nc = N_CYBER if st['cyber'] is not None else 0
    return STEP_PENALTY_SCALE * ((applied[:, :nc] ** 2).sum(1) + 0.09 * (applied[:, nc:] ** 2).sum(1))   # physical part rescaled to a 0.3-step

def build_obs(x_cur, last_applied, last_pred, step):
    oh = np.zeros((len(x_cur), 6), dtype=np.float32); oh[np.arange(len(x_cur)), last_pred] = 1.0
    return np.concatenate([x_cur.astype(np.float32), last_applied.astype(np.float32), oh, np.full((len(x_cur), 1), step / MAX_STEPS, dtype=np.float32)], 1)

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool64, y_pool, oracle, st, seed=None):
        super().__init__(); self.X_pool, self.y_pool, self.oracle, self.st = X_pool64, y_pool, oracle, st; self.rng = np.random.RandomState(seed); self.A = act_dim(st)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.A,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(N_FEAT + self.A + 6 + 1,), dtype=np.float32)
    def _obs(self): return build_obs(self.x_cur, self.last_applied, np.array([self.last_pred]), self.step_count)[0]
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if options and 'x' in options: self.x_orig = np.asarray(options['x'], dtype=np.float64)[None].copy(); self.y_true = options.get('y', 1)
        else:
            idx = self.rng.randint(0, len(self.X_pool)); self.x_orig = self.X_pool[idx][None].copy(); self.y_true = self.y_pool[idx]
        self.x_cur = self.x_orig.copy(); self.raw0, self.rho0 = make_ctx(self.x_orig)
        self.step_count = 0; self.last_applied = np.zeros((1, self.A), dtype=np.float32)
        self.last_pred = int(self.oracle.predict(self.x_cur.astype(np.float32))[0]); return self._obs(), {}
    def step(self, action):
        applied = step_fn(self.x_cur, self.x_orig, self.raw0, self.rho0, np.asarray(action, dtype=np.float64)[None], self.st)
        self.last_applied = applied; self.step_count += 1
        self.last_pred = int(self.oracle.predict(self.x_cur.astype(np.float32))[0]); pen = float(penalty(applied, self.st)[0])
        terminated = truncated = False
        if self.last_pred == 0: reward = EVASION_REWARD - pen; terminated = True
        elif self.step_count >= self.max_steps_(): reward = TIMEOUT_PENALTY - pen; truncated = True
        else: reward = -pen
        return self._obs(), reward, terminated, truncated, {'evaded': self.last_pred == 0}
    def max_steps_(self): return MAX_STEPS

def run_batched(policy, X0_64, oracle, st, mode='ppo', rng=None):
    """One episode per row of X0_64 in parallel (<= MAX_STEPS queries). mode: 'ppo' (deterministic policy) or 'random' (uniform actions)."""
    N = len(X0_64); A = act_dim(st); x_cur = X0_64.copy(); raw0, rho0 = make_ctx(X0_64); last_applied = np.zeros((N, A), dtype=np.float32)
    f32 = lambda Z: np.ascontiguousarray(Z, dtype=np.float32)
    last_pred = np.concatenate([oracle.predict(f32(X0_64[i:i + 65536])) for i in range(0, N, 65536)]).astype(np.int64); clean_pred = last_pred.copy()
    active = np.ones(N, bool); evaded = np.zeros(N, bool); steps = np.zeros(N, np.int64)
    if mode == 'ppo': h = np.zeros((1, N, 64), np.float32); c = np.zeros((1, N, 64), np.float32)
    for step in range(MAX_STEPS):
        idx = np.where(active)[0]
        if len(idx) == 0: break
        obs = build_obs(x_cur[idx], last_applied[idx], last_pred[idx], step)
        if mode == 'ppo':
            act = np.zeros((len(idx), A), np.float32)
            for s in range(0, len(idx), EVAL_CHUNK):
                sl = idx[s:s + EVAL_CHUNK]
                a, (hn, cn) = policy.predict(obs[s:s + EVAL_CHUNK], state=(h[:, sl], c[:, sl]), episode_start=np.full(len(sl), step == 0), deterministic=True)
                act[s:s + EVAL_CHUNK] = a; h[:, sl] = hn; c[:, sl] = cn
        else: act = rng.uniform(-1, 1, size=(len(idx), A)).astype(np.float32)
        sub = x_cur[idx]; applied = step_fn(sub, X0_64[idx], raw0[idx], rho0[idx], act, st); x_cur[idx] = sub; last_applied[idx] = applied
        pred = np.concatenate([oracle.predict(f32(sub[i:i + 65536])) for i in range(0, len(sub), 65536)]).astype(np.int64); last_pred[idx] = pred
        ev = pred == 0; evaded[idx[ev]] = True; steps[idx[ev]] = step + 1; active[idx[ev]] = False
    return {'evaded': evaded, 'steps': steps, 'x_final': x_cur, 'pred_final': last_pred, 'clean_pred': clean_pred, 'raw0': raw0, 'rho0': rho0}

def per_class(vec_bool): return np.array([vec_bool[y_atk_test == c].mean() for c in range(1, 6)])
def mean_steps(res): return np.array([res['steps'][(y_atk_test == c) & res['evaded']].mean() if ((y_atk_test == c) & res['evaded']).any() else np.nan for c in range(1, 6)])
from sklearn.metrics import precision_score, recall_score
BENIGN_PRED = None      # set per seed: the oracle's clean predictions on the benign test rows (the attacker never touches them)
def full_metrics(pred_atk):
    pred = np.empty(len(y_test), dtype=np.int64); pred[y_test == 0] = BENIGN_PRED; pred[atk_te] = pred_atk
    return {'weighted_f1_full': float(f1_score(y_test, pred, average='weighted')), 'macro_f1_full': float(f1_score(y_test, pred, average='macro')),
            'per_class_f1_full': f1_score(y_test, pred, average=None, labels=list(range(6))), 'per_class_recall_full': recall_score(y_test, pred, average=None, labels=list(range(6))),
            'benign_precision': float(precision_score(y_test, pred, labels=[0], average=None)[0])}
def summarise(res, st):
    raw_f = raw_of(res['x_final']); rel = np.abs(raw_f / np.where(res['raw0'] == 0, np.nan, res['raw0']) - 1.0); rel = np.nan_to_num(rel)
    r_f = ratio_of(raw_f); ev = res['evaded']; inside = (r_f >= BAND_LO) & (r_f <= BAND_HI)
    return {'per_class_evasion': per_class(ev), 'overall_evasion': float(ev.mean()), 'mean_steps_to_evade': mean_steps(res),
            **full_metrics(res['pred_final']), 'mean_queries': float(1 + np.where(ev, res['steps'], MAX_STEPS).mean()), 'attack_f1': float(f1_score(y_atk_test, res['pred_final'], average='weighted')),
            'evaded_rows_inside_balance_band': float(inside[ev].mean()) if ev.any() else np.nan,
            'median_rel_change_VI_pct': float(100 * np.median(rel[:, :6].max(1))), 'p99_rel_change_VI_pct': float(100 * np.percentile(rel[:, :6].max(1), 99)),
            'median_rel_change_PQ_pct': float(100 * np.median(rel[:, 6:8].max(1))), 'evaded': ev.copy()}
print('environment, step function and evaluator defined; actions per setting:', {k: act_dim(v) for k, v in SETTINGS.items()})

## 2b. Sanity checks of the step function (run once; cheap)

In [ ]:
rng_chk = np.random.RandomState(0); sel_chk = rng_chk.choice(len(X_atk_test64), size=min(300, len(X_atk_test64)), replace=False)
Xc = X_atk_test64[sel_chk]
for name, st in SETTINGS.items():
    # (1) batched step == one-row-at-a-time step with identical actions
    A = act_dim(st); acts = rng_chk.uniform(-1, 1, size=(MAX_STEPS, len(Xc), A))
    xb = Xc.copy(); r0, p0 = make_ctx(Xc)
    for t in range(MAX_STEPS): step_fn(xb, Xc, r0, p0, acts[t], st)
    xs = Xc.copy()
    for i in range(len(Xc)):
        xi = xs[i:i + 1].copy(); o = Xc[i:i + 1]; ri, pi = make_ctx(o)
        for t in range(MAX_STEPS): step_fn(xi, o, ri, pi, acts[t][i:i + 1], st)
        xs[i] = xi[0]
    eq = float(np.abs(xb - xs).max())
    # (2) invariants of the final state
    raw_f = raw_of(xb); rel = np.abs(raw_f[:, :6] / r0[:, :6] - 1).max()
    msg = f'  {name:<22} batch==single max diff {eq:.1e} | max rel change of V,I {100 * rel:.3f}% (budget {100 * st["eps_rel"]:.1f}%)'
    if st['balance']:
        dev = np.abs(ratio_of(raw_f) / p0 - 1); msg += f' | balance drift vs original: median {np.median(dev):.1e}, p99 {np.percentile(dev, 99):.1e}, max {dev.max():.1e}'
    if st['cyber'] is not None: msg += f' | max cyber move {np.abs(xb[:, CYBER_IDX] - Xc[:, CYBER_IDX]).max():.3f} sigma (budget {st["cyber"]})'
    print(msg); assert eq < 1e-9 and rel <= st['eps_rel'] + 1e-9
print('step-function checks passed')

## 3. Training / loading an agent (resumable, fingerprinted)

In [ ]:
def fp_of(*parts): return hashlib.sha256(repr(parts).encode()).hexdigest()[:16]
def model_fp(oracle, X_probe): return hashlib.sha256(np.asarray(oracle.predict(X_probe)).tobytes()).hexdigest()[:16]
def latest_ckpt(d):
    if not os.path.isdir(d): return None
    c = [f for f in os.listdir(d) if f.endswith('_steps.zip')]
    if not c: return None
    best = max(c, key=lambda f: int(f.rsplit('_steps.zip', 1)[0].rsplit('_', 1)[1])); return os.path.join(d, best)

def get_agent(sname, seed, oracle):
    st = SETTINGS[sname]; name = f'ppo_attack2d_{sname}_{PIPELINE_VERSION}_seed{seed}'
    fp = fp_of(model_fp(oracle, X_atk_train[:300]), EPS_BOUND, MAX_STEPS, EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY, TOTAL_TIMESTEPS, sorted(st.items(), key=str), seed, 'nb4d-v1')
    if os.path.exists(name + '.zip') and os.path.exists(name + '.fp') and open(name + '.fp').read() == fp:
        print(f'    [{sname}/{seed}] finished agent found (fingerprint matches) -- loaded'); return RecurrentPPO.load(name), 0.0
    ckdir = f'_ckpt_{name}'; os.makedirs(ckdir, exist_ok=True); fpf = os.path.join(ckdir, '_fp.txt')
    env = IDSEvasionEnv(X_atk_train64, y_atk_train, oracle, st, seed=seed); ck = latest_ckpt(ckdir)
    if ck and os.path.exists(fpf) and open(fpf).read() == fp:
        model = RecurrentPPO.load(ck, env=env); remaining = TOTAL_TIMESTEPS - model.num_timesteps
        print(f'    [{sname}/{seed}] resuming from {model.num_timesteps}/{TOTAL_TIMESTEPS}')
    else:
        shutil.rmtree(ckdir, ignore_errors=True); os.makedirs(ckdir); open(fpf, 'w').write(fp)
        model = RecurrentPPO('MlpLstmPolicy', env, n_steps=128, batch_size=64, learning_rate=3e-4, gamma=0.99, seed=seed, verbose=0,
                             policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1), device='cpu'); remaining = TOTAL_TIMESTEPS
    t0 = time.time()
    if remaining > 0:
        model.learn(total_timesteps=remaining, reset_num_timesteps=False, callback=CheckpointCallback(save_freq=CKPT_FREQ, save_path=ckdir, name_prefix='ckpt'), progress_bar=False)
    model.save(name); open(name + '.fp', 'w').write(fp); shutil.rmtree(ckdir, ignore_errors=True)
    return model, time.time() - t0

## 4. Per-seed run

In [ ]:
if SMOKE_TEST: TOTAL_TIMESTEPS = 1024
R = {}; INFO = {}; t_all = time.time()          # R[seed][(setting, 'PPO'|'Random')] -> summary
for seed in SEEDS:
    pf = f'attack2d_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = fp_of(best_params, EPS_BOUND, MAX_STEPS, TOTAL_TIMESTEPS, [(k, sorted(SETTINGS[k].items(), key=str)) for k in SETTINGS_TO_RUN], seed, X_atk_test.shape, float(X_atk_test[:5].sum()), 'nb4d-v1')
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp and all('weighted_f1_full' in v for v in saved['R'].values()): R[seed] = saved['R']; INFO[seed] = saved['info']; print(f'seed {seed}: result file found (fingerprint matches) -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False)
    f1_fit = f1_score(y_test, clf.predict(X_test), average='weighted'); clf.set_params(device='cpu', n_jobs=-1); oracle = Oracle(clf)
    _cp = clf.predict(X_test); BENIGN_PRED = _cp[y_test == 0]; INFO_CLEAN = float(f1_score(y_test, _cp, average='weighted'))
    INFO[seed] = {'clean_f1_fit_backend': float(f1_fit), 'clean_f1_cpu_oracle': float(f1_score(y_test, clf.predict(X_test), average='weighted'))}
    R[seed] = {}
    for sname in SETTINGS_TO_RUN:
        st = SETTINGS[sname]; agent, tt = get_agent(sname, seed, oracle)
        R[seed][(sname, 'PPO')] = summarise(run_batched(agent, X_atk_test64, oracle, st, mode='ppo'), st)
        R[seed][(sname, 'Random')] = summarise(run_batched(None, X_atk_test64, oracle, st, mode='random', rng=np.random.RandomState(seed + 900)), st)
        r, rr = R[seed][(sname, 'PPO')], R[seed][(sname, 'Random')]
        print(f'  {sname:<22} PPO per-class {np.round(r["per_class_evasion"], 3).tolist()} overall {r["overall_evasion"]:.3f} (random {rr["overall_evasion"]:.3f}) | full-test F1 {r["weighted_f1_full"]:.4f} (clean {INFO_CLEAN:.4f}) | queries {r["mean_queries"]:.2f} | inside balance band {r["evaded_rows_inside_balance_band"]:.3f} | train {tt / 60:.1f} min', flush=True)
    joblib.dump({'fp': cfg_fp, 'R': R[seed], 'info': INFO[seed]}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. Results: per-class evasion (mean +/- std over seeds), with the cyber-only references from NB4 / NB4b

In [ ]:
atk_names = class_names[1:]
def have(key): return [s for s in SEEDS if key in R[s]]
def arr(key, field): return np.stack([np.asarray(R[s][key][field]) for s in have(key)])
sdv = lambda x: x.std(0, ddof=1) if len(x) > 1 else np.zeros(x.shape[1:])
rows = []
for sname in SETTINGS_TO_RUN:
    for who in ['PPO', 'Random']:
        key = (sname, who)
        if not have(key): continue
        a = arr(key, 'per_class_evasion'); o = arr(key, 'overall_evasion'); n = len(have(key)); st = SETTINGS[sname]
        row = {'setting': sname, 'attacker': who, 'cyber (sigma)': st['cyber'] if st['cyber'] is not None else '-', 'physical (rel.)': f'{100 * st["eps_rel"]:.0f}%', 'balance': 'yes' if st['balance'] else 'no', 'seeds': n}
        row.update({c: f'{a[:, i].mean():.3f} +/- {sdv(a)[i]:.3f}' for i, c in enumerate(atk_names)}); row['ALL'] = f'{o.mean():.3f} +/- {(o.std(ddof=1) if n > 1 else 0):.3f}'
        row['queries'] = f'{arr(key, "mean_queries").mean():.2f}'; row['full-test weighted F1'] = f'{arr(key, "weighted_f1_full").mean():.4f}'; row['benign precision'] = f'{arr(key, "benign_precision").mean():.3f}'; rows.append(row)
res_table = pd.DataFrame(rows); pd.set_option('display.width', 300); print(res_table.to_string(index=False))

# cyber-only references (NB4b / NB4), same agents and oracle protocol
ref_rows = []
def load_ref(budget, label, method='PPO'):
    vals = []
    for s in SEEDS:
        f = f'attack2b_seed{s}_{PIPELINE_VERSION}.pkl'
        if not os.path.exists(f): return
        d = joblib.load(f)['R']; k = (method, budget)
        if k in d: vals.append(d[k]['per_class_evasion'])
        elif budget == 0.1 and os.path.exists(f'attack2_seed{s}_{PIPELINE_VERSION}.pkl'): vals.append(joblib.load(f'attack2_seed{s}_{PIPELINE_VERSION}.pkl')['R']['PPO_budgeted']['per_class_evasion'])
    if len(vals) == len(SEEDS):
        a = np.stack(vals); row = {'reference': label}; row.update({c: f'{a[:, i].mean():.3f}' for i, c in enumerate(atk_names)}); row['ALL'] = f'{a.mean(1).mean():.3f}'; ref_rows.append(row)
for b in [0.1, 0.5, 1.0]: load_ref(b, f'cyber-only PPO at {b} sigma (NB4/NB4b)')
if ref_rows: print('\nCyber-only references:'); print(pd.DataFrame(ref_rows).to_string(index=False))

print('\nPer-seed ransomware / reverseshell evasion (PPO):')
for sname in SETTINGS_TO_RUN:
    key = (sname, 'PPO')
    if have(key): print(f'  {sname:<22} ransomware {np.round(arr(key, "per_class_evasion")[:, 3], 3).tolist()}  reverseshell {np.round(arr(key, "per_class_evasion")[:, 4], 3).tolist()}')

## 6. Are the adversarial rows physically consistent? (balance band, size of the changes)

In [ ]:
real_rows = []
for sname in SETTINGS_TO_RUN:
    key = (sname, 'PPO')
    if not have(key): continue
    g = lambda f: np.nanmean(arr(key, f))
    real_rows.append({'setting': sname, 'evaded rows inside genuine balance band': f'{g("evaded_rows_inside_balance_band"):.3f}',
                      'median max rel. change of V,I (%)': f'{g("median_rel_change_VI_pct"):.2f}', 'p99 (%)': f'{g("p99_rel_change_VI_pct"):.2f}', 'median rel. change of P,Q (%)': f'{g("median_rel_change_PQ_pct"):.2f}'})
real_table = pd.DataFrame(real_rows); print(real_table.to_string(index=False))
print(f'\nGenuine band = [{BAND_LO:.5f}, {BAND_HI:.5f}] for S / sum(V*I). Setting A has no balance constraint, so its share inside the band shows how far an unconstrained physical forger drifts from the identity; the balance settings should be close to 1 (small drift only from range clipping).')

## 7. Paired tests (same target per seed; n = 5; Holm-adjusted over the 5 classes)

In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
def ptest(A, B):
    d = A - B
    if len(d) < 2: return float('nan')
    if d.std(ddof=1) == 0: return 1.0 if d.mean() == 0 else 0.0
    return float(stats.ttest_rel(A, B).pvalue)
def compare(k1, k2, label):
    if not (len(have(k1)) == len(SEEDS) and len(have(k2)) == len(SEEDS)): return []
    A, B = arr(k1, 'per_class_evasion'), arr(k2, 'per_class_evasion'); raw = np.array([ptest(A[:, i], B[:, i]) for i in range(5)]); adj = holm(np.nan_to_num(raw, nan=1.0)); oa, ob = arr(k1, 'overall_evasion'), arr(k2, 'overall_evasion')
    out = [{'comparison': label, 'class': c, 'diff': f'{(A - B)[:, i].mean():+.3f}', 'p_raw': f'{raw[i]:.4f}', 'p_holm': f'{adj[i]:.4f}'} for i, c in enumerate(atk_names)]
    out.append({'comparison': label, 'class': 'ALL', 'diff': f'{(oa - ob).mean():+.3f}', 'p_raw': f'{ptest(oa, ob):.4f}', 'p_holm': '-'}); return out
stat_rows = []
for sname in SETTINGS_TO_RUN: stat_rows += compare((sname, 'PPO'), (sname, 'Random'), f'{sname}: PPO minus random')
if 'A_phys5_nobalance' in SETTINGS_TO_RUN and 'B_phys5_balance' in SETTINGS_TO_RUN: stat_rows += compare(('A_phys5_nobalance', 'PPO'), ('B_phys5_balance', 'PPO'), 'cost of the balance constraint: A minus B (PPO)')
stat_table = pd.DataFrame(stat_rows); print(stat_table.to_string(index=False) if len(stat_table) else 'no paired tests (incomplete seeds)')

## 8. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8)); w = 0.8 / max(1, len(SETTINGS_TO_RUN))
for k, sname in enumerate(SETTINGS_TO_RUN):
    if not have((sname, 'PPO')): continue
    a = arr((sname, 'PPO'), 'per_class_evasion'); ax[0].bar(np.arange(5) + (k - (len(SETTINGS_TO_RUN) - 1) / 2) * w, a.mean(0), w, yerr=sdv(a), capsize=2, label=sname)
ax[0].set_xticks(np.arange(5)); ax[0].set_xticklabels(atk_names, rotation=15); ax[0].set_ylabel('PPO evasion rate'); ax[0].set_title('Per-class evasion with a physical-layer attacker'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3, axis='y')
names = [s for s in SETTINGS_TO_RUN if have((s, 'PPO'))]; xs = np.arange(len(names))
ax[1].bar(xs - 0.2, [arr((s, 'PPO'), 'overall_evasion').mean() for s in names], 0.4, yerr=[arr((s, 'PPO'), 'overall_evasion').std(ddof=1) if len(have((s, 'PPO'))) > 1 else 0 for s in names], capsize=2, label='PPO')
ax[1].bar(xs + 0.2, [arr((s, 'Random'), 'overall_evasion').mean() for s in names], 0.4, label='random'); ax[1].set_xticks(xs); ax[1].set_xticklabels(names, rotation=15, fontsize=8)
ax[1].set_ylabel('overall evasion'); ax[1].set_title('Overall evasion, PPO vs random'); ax[1].legend(); ax[1].grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'attack2d_physical_{PIPELINE_VERSION}.png', dpi=150); plt.show()

joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'SETTINGS': SETTINGS, 'SETTINGS_TO_RUN': SETTINGS_TO_RUN, 'class_names': class_names, 'res_table': res_table, 'real_table': real_table, 'stat_table': stat_table,
             'balance_band': (float(BAND_LO), float(BAND_HI)), 'per_seed': {s: {f'{k[0]}|{k[1]}': {kk: vv for kk, vv in v.items() if kk != 'evaded'} for k, v in R[s].items()} for s in SEEDS}, 'info': INFO},
            f'attack2d_results_{PIPELINE_VERSION}.pkl')
res_table.to_csv(f'attack2d_results_table_{PIPELINE_VERSION}.csv', index=False); real_table.to_csv(f'attack2d_realism_{PIPELINE_VERSION}.csv', index=False); stat_table.to_csv(f'attack2d_paired_tests_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB4d', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'settings': SETTINGS, 'timesteps': TOTAL_TIMESTEPS, 'sb3_contrib': sb3_contrib.__version__, 'torch': torch.__version__,
           'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb4d_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack2d_results_{PIPELINE_VERSION}.pkl', f'attack2d_results_table_{PIPELINE_VERSION}.csv', f'attack2d_realism_{PIPELINE_VERSION}.csv', f'attack2d_paired_tests_{PIPELINE_VERSION}.csv', f'attack2d_physical_{PIPELINE_VERSION}.png', f'nb4d_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)